#### Setup LangChain

pip install -U langchain langchain_community langchain-openai

In [23]:
import langchain
langchain.__version__

'1.2.15'


#### Credentials

In [2]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

In [3]:
if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")

os.environ["LANGSMITH_TRACING"] = "true"

#### Instantiation

In [ ]:
import os
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
    # stream_usage=True,
    # temperature=None,
    # max_tokens=None,
    # timeout=None,
    # reasoning_effort="low",
    # max_retries=2,
    # organization="...",
    # other params...
)


'\nfor chunk in model.stream(input_text):\n    print("\n")\n    print(chunk, end="|")\n'

#### Invocation : Create Prompt

In [19]:
messages = [
    ("system", "You are a helpful assistant that translates English to Hindi. Translate the user sentence."),
    ("human", "I love programming."),
]

ai_msg = model.invoke("translate I love programming to Hindi")
ai_msg

AIMessage(content='**Hindi translation**  \n- “I love programming.” → **मुझे प्रोग्रामिंग से प्यार है।**\n\n**Transliteration**  \n- "Mujhe programming se pyaar hai."\n\n*(If you want a gender‑specific version, you can say)*  \n- Male: “मैं प्रोग्रामिंग से प्यार करता हूँ।”  \n- Female: “मैं प्रोग्रामिंग से प्यार करती हूँ।”', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 214, 'prompt_tokens': 77, 'total_tokens': 291, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 119, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.043921548, 'prompt_time': 0.003765596, 'completion_time': 0.236036295, 'total_time': 0.239801891}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'id': 'chatcmpl-1c8edc5e-1cd2-4989-88f8-706a0a5d3b57', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='lc

```json
AIMessage(
    content="J'adore la programmation.",
    response_metadata={
        "token_usage": {
            "completion_tokens": 5,
            "prompt_tokens": 31,
            "total_tokens": 36,
        },
        "model_name": "gpt-4o",
        "system_fingerprint": "fp_43dfabdef1",
        "finish_reason": "stop",
        "logprobs": None,
    },
    id="run-012cffe2-5d3d-424d-83b5-51c6d4a593d1-0",
    usage_metadata={"input_tokens": 31, "output_tokens": 5, "total_tokens": 36},
)
```

In [20]:
print(ai_msg.text)

**Hindi translation**  
- “I love programming.” → **मुझे प्रोग्रामिंग से प्यार है।**

**Transliteration**  
- "Mujhe programming se pyaar hai."

*(If you want a gender‑specific version, you can say)*  
- Male: “मैं प्रोग्रामिंग से प्यार करता हूँ।”  
- Female: “मैं प्रोग्रामिंग से प्यार करती हूँ।”


### Stream

In [21]:
for chunk in model.stream(messages):
    chunk
    print(chunk.text, end="")

मुझे प्रोग्रामिंग से प्यार है.

To collect the full message, you can concatenate the chunks:

In [22]:
stream = model.stream(messages)
full = next(stream)
for chunk in stream:
    full += chunk

In [ ]:
full = AIMessageChunk(
    content="J'adore la programmation.",
    response_metadata={"finish_reason": "stop"},
    id="run-bf917526-7f58-4683-84f7-36a6b671d140",
)

`Async` : Asynchronous equivalents of invoke, stream, and batch are also available:

In [ ]:
# Invoke
await model.ainvoke(messages)

# Stream
async for chunk in (await model.astream(messages))

# Batch
await model.abatch([messages])

Tool calling

In [ ]:
from pydantic import BaseModel, Field

class GetWeather(BaseModel):
    '''Get the current weather in a given location'''
    location: str = Field(..., description="The city and state, e.g. San Francisco, CA")

class GetPopulation(BaseModel):
    '''Get the current population for a given location'''
    population: float = Field(..., description="The city and population, e.g. San Francisco, 10.55")

model_with_tools = model.bind_tools(
    [GetWeather, GetPopulation]
    # strict = True  # Enforce tool args schema is respected
)

ai_msg = model_with_tools.invoke(
    "Which city is hotter today and which is bigger: LA or NY?"
)

ai_msg.tool_calls[
    {
        "name": "GetWeather",
        "args": {"location": "Los Angeles, CA"},
        "id": "call_6XswGD5Pqk8Tt5atYr7tfenU",
    },
    {
        "name": "GetWeather",
        "args": {"location": "New York, NY"},
        "id": "call_ZVL15vA8Y7kXqOy3dtmQgeCi",
    },
    {
        "name": "GetPopulation",
        "args": {"location": "Los Angeles, CA"},
        "id": "call_49CFW8zqC9W7mh7hbMLSIrXw",
    },
    {
        "name": "GetPopulation",
        "args": {"location": "New York, NY"},
        "id": "call_6ghfKxV264jEfe1mRIkS3PE7",
    },
]

[{'name': 'GetWeather',
  'args': {'location': 'Los Angeles, CA'},
  'id': 'fc_82d725d7-2968-4abc-ba0f-1cceec1f7590',
  'type': 'tool_call'}]

Parallel tool calls

In [ ]:
ai_msg = model_with_tools.invoke(
    "What is the weather in LA and NY?", parallel_tool_calls=False
)
ai_msg.tool_calls[
    {
        "name": "GetWeather",
        "args": {"location": "Los Angeles, CA"},
        "id": "call_4OoY0ZR99iEvC7fevsH8Uhtz",
    }
]

Built-in (server-side) tools

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="...", output_version="responses/v1")

tool = {"type": "web_search"}
model_with_tools = model.bind_tools([tool])

response = model_with_tools.invoke("What was a positive news story from today?")
response.content[
    {
        "type": "text",
        "text": "Today, a heartwarming story emerged from ...",
        "annotations": [
            {
                "end_index": 778,
                "start_index": 682,
                "title": "Title of story",
                "type": "url_citation",
                "url": "<url of story>",
            }
        ],
    }
]